# TiDE (deep learning) + LightGBM ensemble — v2

**What this is.** A global [TiDE](https://arxiv.org/abs/2304.08424) neural forecaster trained
across all 1,782 store×family series via [`darts`](https://unit8co.github.io/darts/), blended
with this project's production LightGBM.

**Why a neural net specifically.** An XGBoost/LightGBM ensemble was tried and rejected: the two
GBDT libraries' residuals correlated at **0.974–0.987** — different tree-growth algorithm, same
mistakes, nothing to diversify. v1 of this notebook confirmed the counterpart: TiDE's residuals
correlate with LightGBM's at only **0.770**. A neural network genuinely does err differently
here. That premise is settled; what remains is whether it can be made *good* enough to earn
weight in a blend.

## What v1 got wrong, and what changed

v1 scored **0.66126** on the leaderboard — the worst submission in this project — for reasons
that were mostly not about the model:

| Problem | v1 | v2 |
|---|---|---|
| **Refit trained 0 epochs** (`max_epochs=0`), so an *untrained* net entered the blend and halved chain-wide volume (6.4M vs 12.9M units) | fatal | fixed, plus two guards |
| LightGBM half deliberately handicapped (39 features, 1 seed → holdout 0.40219) | could never beat 0.42074 | full 60 features, 5 seeds, matches production |
| TiDE fed raw `log1p` across series spanning 5 orders of magnitude | solo 0.50711 | reversible instance norm |
| Lookback 90 days, hidden 128, 1+1 layers | 237 K params | 120 days, hidden 256, 2+2 layers |

**The measurement that decides this.** v1's blend sweep, computed with a *correctly* trained
TiDE, gave a best gain of **+0.0002** at `w(lgbm)=0.9` — roughly 20× below this project's
0.0039 seed-noise floor, i.e. nothing. For v2 to be worth submitting, TiDE's solo score has to
land near LightGBM's (~0.39–0.42), not near the naive baseline (0.5206). **If TiDE solo comes
back above ~0.45, the honest call is to stop — the blend will not rescue it, and the sweep will
just pick `w=1.0`.**

**Runtime ≈ 2 hours** on a T4. Requires Settings → Accelerator → **GPU**, and Settings →
Internet → **On** (for the `darts` install).

In [1]:
# darts is not preinstalled on Kaggle; torch is. Installing it without care breaks the image.
#
# The hazard: Kaggle's pandas, scipy and lightgbm are BINARY wheels compiled against the exact
# numpy in the image. If pip changes numpy by even a minor version, those break at import with
# errors like "numpy.dtype size changed" or "cannot import name '_center' from
# numpy._core.umath". Two earlier attempts failed exactly this way -- once downgraded to 1.26.4
# (by the full `darts`, which pulls prophet/statsforecast/pmdarima), once to 2.0.2 (by a loose
# `numpy>=2.0` pin, when the image actually shipped 2.5.2).
#
# Interactively you can paper over this by restarting the kernel after installing. A committed
# "Save & Run All" runs every cell in ONE process with no restart available, so the only safe
# approach is to never let numpy change at all: read the image's exact version and pin to it,
# so pip either honours it or fails loudly in 30s. `u8darts[torch]` is the slim variant --
# only the torch-based models, of which TiDE is one -- avoiding the heaviest numpy pins.
import numpy, subprocess, sys

NP_BEFORE = numpy.__version__
print(f"image numpy: {NP_BEFORE} -- pinning to this exact version")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "u8darts[torch]", f"numpy=={NP_BEFORE}"])

import importlib
importlib.reload(numpy)
assert numpy.__version__ == NP_BEFORE, (
    f"numpy changed {NP_BEFORE} -> {numpy.__version__} despite the pin. Any change breaks the "
    "image's precompiled scipy/pandas/lightgbm on a committed run. Do not proceed."
)
print(f"numpy still {numpy.__version__} -- image intact")

import gc, time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
from darts import TimeSeries
from darts.models import TiDEModel
from pytorch_lightning.callbacks import EarlyStopping

pd.set_option("display.width", 120)

GPU = torch.cuda.is_available()
print(f"GPU available: {GPU}" + (f"  ({torch.cuda.get_device_name(0)})" if GPU else ""))
assert GPU, ("No GPU. Set Settings -> Accelerator -> GPU and restart -- TiDE on CPU here "
             "would take many hours.")

image numpy: 2.0.2 -- pinning to this exact version
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.8/418.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.3 MB/s eta 0:00:00


/usr/lib/python3.12/importlib/__init__.py:131: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  _bootstrap._exec(spec, module)


numpy still 2.0.2 -- image intact


The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


GPU available: True  (Tesla T4)


In [2]:
COMP = "store-sales-time-series-forecasting"
REQUIRED = {"train.csv", "test.csv", "stores.csv", "holidays_events.csv"}

def has_data(p: Path) -> bool:
    try:
        return p.is_dir() and REQUIRED.issubset({f.name for f in p.iterdir() if f.is_file()})
    except OSError:
        return False

def find_data() -> Path:
    root = Path("/kaggle/input")
    if root.is_dir():
        for cand in [root / COMP, *sorted(d for d in root.iterdir() if d.is_dir())]:
            if has_data(cand):
                return cand
    for cand in (Path("data"), Path("../data"), Path("../../data")):
        if has_data(cand):
            return cand
    # Fallback if the competition is not attached as an input. Attaching it in the editor
    # (+ Add Input -> Competitions -> store-sales-time-series-forecasting) is preferable --
    # it is also what makes "Submit to Competition" work -- but this keeps the notebook
    # runnable if someone forgets.
    try:
        import kagglehub
        got = Path(kagglehub.competition_download(COMP))
        if has_data(got):
            return got
        for sub in got.rglob("*"):
            if has_data(sub):
                return sub
    except Exception as exc:
        print(f"kagglehub fallback failed: {exc}")
    raise FileNotFoundError(
        f"Could not find {sorted(REQUIRED)}. In the Kaggle editor, use + Add Input -> "
        "Competitions -> store-sales-time-series-forecasting to attach the data."
    )

DATA = find_data()
ON_KAGGLE = Path("/kaggle/working").is_dir()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("submissions/tide_ensemble")
OUT.mkdir(parents=True, exist_ok=True)

HORIZON = 16
MODEL_START = pd.Timestamp("2015-01-01")
EQ_START, EQ_END = pd.Timestamp("2016-04-16"), pd.Timestamp("2016-04-22")

# The validated production model's chain-wide 16-day total. Any new submission whose volume
# departs materially from this is wrong -- this exact check would have caught v1's untrained
# refit (6,382,939 units) in seconds, before uploading.
VALIDATED_VOLUME = 12_932_032

DTYPES = {"store_nbr": "int8", "family": "category", "onpromotion": "int32", "sales": "float32"}
train  = pd.read_csv(DATA / "train.csv", parse_dates=["date"], dtype=DTYPES)
test   = pd.read_csv(DATA / "test.csv",  parse_dates=["date"], dtype=DTYPES)
stores = pd.read_csv(DATA / "stores.csv", dtype={"store_nbr": "int8"})
hol    = pd.read_csv(DATA / "holidays_events.csv", parse_dates=["date"])

TRAIN_END   = train.date.max()
VALID_START = TRAIN_END - pd.Timedelta(days=HORIZON - 1)
print(f"train {train.date.min():%Y-%m-%d} -> {TRAIN_END:%Y-%m-%d}, "
      f"test {test.date.min():%Y-%m-%d} -> {test.date.max():%Y-%m-%d}")

train 2013-01-01 -> 2017-08-15, test 2017-08-16 -> 2017-08-31


---
## Panel construction — identical to the production notebook

Wide panel, Christmas restored as zero-sales days, April-2016 earthquake week replaced by each series' same-weekday median from the surrounding 8 weeks. Copied verbatim from `kaggle_store_sales_submission.ipynb` so both halves of the blend train on the same cleaned history.

In [3]:
t0 = time.time()
full_idx = pd.date_range(train.date.min(), test.date.max(), freq="D")

both = pd.concat([train.drop(columns="sales"), test], ignore_index=True)
both["family"] = both.family.astype(str)

sales_w = (train.assign(family=train.family.astype(str))
           .pivot(index="date", columns=["store_nbr", "family"], values="sales")
           .reindex(full_idx).sort_index(axis=1))
promo_w = (both.pivot(index="date", columns=["store_nbr", "family"], values="onpromotion")
           .reindex(full_idx).sort_index(axis=1).fillna(0.0))

xmas = pd.DatetimeIndex([d for d in full_idx
                         if d <= TRAIN_END and d not in set(train.date.unique())])
sales_w.loc[xmas] = 0.0
assert sales_w.loc[:TRAIN_END].isna().sum().sum() == 0

SERIES = sales_w.columns
n_s = len(SERIES)

def repair_window(W, start, end, halo_weeks=8):
    out = W.copy()
    win = pd.date_range(start, end)
    ctx = W.loc[start - pd.Timedelta(weeks=halo_weeks): end + pd.Timedelta(weeks=halo_weeks)]
    ctx = ctx.drop(index=win, errors="ignore")
    by_dow = ctx.groupby(ctx.index.dayofweek).median()
    for d in win:
        out.loc[d] = by_dow.loc[d.dayofweek].values
    return out

sales_w = repair_window(sales_w, EQ_START, EQ_END)
eq_dates = pd.date_range(EQ_START, EQ_END)
print(f"panel built: {sales_w.shape}, {n_s} series  ({time.time()-t0:.0f}s)")

panel built: (1704, 1782), 1782 series  (36s)


In [4]:
h = hol.copy()
h = h[~((h.type == "Holiday") & (h.transferred))]
h.loc[h.type == "Transfer", "type"] = "Holiday"
work_days = set(h.loc[h.type == "Work Day", "date"])
h = h[h.type != "Work Day"]
events = h[h.type == "Event"]
h = h[h.type != "Event"]

nat = pd.DatetimeIndex(sorted(set(h.loc[h.locale == "National", "date"])))
nat_name = (h[h.locale == "National"].drop_duplicates("date")[["date", "description"]]
            .rename(columns={"description": "nat_hol_name"}))
loc_name = (h[h.locale == "Local"].drop_duplicates(["date", "locale_name"])
            [["date", "locale_name", "description"]]
            .rename(columns={"locale_name": "city", "description": "loc_hol_name"}))

geo = stores.set_index("store_nbr")[["city", "state", "type", "cluster"]]
geo.columns = ["city", "state", "store_type", "cluster"]
assert not (set(loc_name.city) - set(stores.city))

pos = np.searchsorted(nat.values, full_idx.values)
prev_d = np.where(pos > 0, (full_idx.values - nat.values[np.maximum(pos - 1, 0)])
                  / np.timedelta64(1, "D"), 999)
next_d = np.where(pos < len(nat), (nat.values[np.minimum(pos, len(nat) - 1)] - full_idx.values)
                  / np.timedelta64(1, "D"), 999)
cal_hol = pd.DataFrame({"date": full_idx,
                        "work_day": full_idx.isin(work_days).astype("int8"),
                        "is_event": full_idx.isin(set(events.date)).astype("int8"),
                        "days_since_nat": np.clip(prev_d, 0, 30).astype("int16"),
                        "days_to_nat": np.clip(next_d, 0, 30).astype("int16")})
print(f"{len(nat_name)} national + {len(loc_name)} local holiday dates")

102 national + 147 local holiday dates


---
## LightGBM half — the full production configuration

All 60 features and the 5-seed log-space ensemble, exactly as in `kaggle_store_sales_submission.ipynb` (holdout 0.39081, LB 0.42074). v1 used a trimmed 39-feature single-seed model, which meant the blend was being measured against something weaker than what actually ships — a comparison that could not have produced a usable answer even had the refit worked.

In [5]:
L = np.log1p(sales_w).astype("float32")
P = np.log1p(promo_w).astype("float32")
ZERO = (sales_w == 0).astype("float32")

feats = {}
for k in (16, 17, 18, 19, 20, 21, 22, 28, 35, 49, 63):
    feats[f"lag_{k}"] = L.shift(k)

base = L.shift(HORIZON)
for w in (7, 14, 28, 56, 112):
    feats[f"rmean_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).mean()
for w in (14, 28):
    feats[f"rstd_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).std()
feats["rmax_28"] = base.rolling(28, min_periods=7).max()

feats["dow_mean_4"] = sum(L.shift(k) for k in (21, 28, 35, 42)) / 4
feats["dow_mean_8"] = sum(L.shift(k) for k in range(21, 21 + 7 * 8, 7)) / 8

feats["zfrac_28"] = ZERO.shift(HORIZON).rolling(28, min_periods=7).mean()
feats["zfrac_112"] = ZERO.shift(HORIZON).rolling(112, min_periods=28).mean()

A = (sales_w.to_numpy() > 0)
gap = np.empty(A.shape, dtype="float32")
last = np.full(A.shape[1], -999.0)
for i in range(A.shape[0]):
    gap[i] = i - last
    last = np.where(A[i], float(i), last)
feats["days_since_sale"] = pd.DataFrame(np.minimum(gap, 999.0), index=sales_w.index,
                                        columns=SERIES).shift(HORIZON)

feats["promo"] = P
feats["promo_rmean_7"] = P.rolling(7, min_periods=1).mean()
feats["promo_rmean_28"] = P.rolling(28, min_periods=3).mean()
feats["promo_lag_16"] = P.shift(HORIZON)
feats["promo_rel_112"] = P - P.rolling(112, min_periods=14).mean()
feats["promo_rel_28"] = P - P.rolling(28, min_periods=5).mean()
for k in (1, 2, 3, 7):
    feats[f"promo_lead_{k}"] = P.shift(-k)
feats["promo_fwd7"] = P.shift(-6).rolling(7, min_periods=1).mean()

chain = P.mean(axis=1)
chain_rel = chain - chain.rolling(112, min_periods=14).mean()
ones = np.ones(n_s, dtype="float32")
feats["promo_chain_level"] = pd.DataFrame(np.outer(chain.to_numpy(dtype="float32"), ones),
                                          index=full_idx, columns=SERIES)
feats["promo_chain_rel"] = pd.DataFrame(np.outer(chain_rel.to_numpy(dtype="float32"), ones),
                                        index=full_idx, columns=SERIES)
del chain, chain_rel; gc.collect()

promo_raw = np.expm1(P)
fam_plog = np.log1p(promo_raw.T.groupby(level=1).sum().T)
store_plog = np.log1p(promo_raw.T.groupby(level=0).sum().T)
def _bcast(agg, level):
    return agg[SERIES.get_level_values(level)].set_axis(SERIES, axis=1)
feats["fam_promo_rel"] = _bcast(fam_plog - fam_plog.rolling(112, min_periods=14).mean(), 1)
feats["fam_promo_fwd7"] = _bcast(fam_plog.shift(-6).rolling(7, min_periods=1).mean()
                                 - fam_plog.rolling(112, min_periods=14).mean(), 1)
feats["store_promo_rel"] = _bcast(store_plog - store_plog.rolling(112, min_periods=14).mean(), 0)
del promo_raw, fam_plog, store_plog; gc.collect()

print(f"{len(feats)} panel features built ({time.time()-t0:.0f}s)")

40 panel features built (39s)


In [6]:
mask = full_idx >= MODEL_START
dates_sel = full_idx[mask]

df = pd.DataFrame({
    "date": np.repeat(dates_sel.values, n_s),
    "store_nbr": np.tile(SERIES.get_level_values(0).to_numpy(), len(dates_sel)),
    "family": np.tile(SERIES.get_level_values(1).to_numpy(), len(dates_sel)),
})
for name, W in feats.items():
    df[name] = W.to_numpy(dtype="float32")[mask].ravel()
df["target"] = L.to_numpy(dtype="float32")[mask].ravel()
del feats; gc.collect()

df = df.join(geo, on="store_nbr")
d = df.date
df["dow"] = d.dt.dayofweek.astype("int8")
df["day"] = d.dt.day.astype("int8")
df["month"] = d.dt.month.astype("int8")
df["year"] = d.dt.year.astype("int16")
df["dayofyear"] = d.dt.dayofyear.astype("int16")
df["is_weekend"] = (df.dow >= 5).astype("int8")
df["days_to_month_end"] = (d.dt.days_in_month - d.dt.day).astype("int8")
df["payday_window"] = (d.dt.day.isin([15, 16, 17, 1, 2, 3]) |
                       (df.days_to_month_end <= 1)).astype("int8")

df = df.merge(cal_hol, on="date", how="left")
df = df.merge(nat_name, on="date", how="left")
df = df.merge(loc_name, on=["date", "city"], how="left")
df["nat_hol_name"] = df.nat_hol_name.fillna("none")
df["loc_hol_name"] = df.loc_hol_name.fillna("none")

for c in ("family", "city", "state", "store_type", "nat_hol_name", "loc_hol_name"):
    df[c] = df[c].astype("category")
df["store_nbr"] = df.store_nbr.astype("int16")
df["cluster"] = df.cluster.astype("int16")

FEATURES = [c for c in df.columns if c not in ("date", "target")]
CATS = ["family", "city", "state", "store_type", "nat_hol_name", "loc_hol_name"]
print(f"design matrix: {df.shape[0]:,} rows x {len(FEATURES)} features "
      f"(production is 60)  ({time.time()-t0:.0f}s)")

design matrix: 1,735,668 rows x 60 features (production is 60)  (43s)


In [7]:
eq_row = df.date.isin(eq_dates)
tr_m = (df.date < VALID_START) & df.target.notna() & ~eq_row
va_m = (df.date >= VALID_START) & (df.date <= TRAIN_END)
te_m = df.date > TRAIN_END

PARAMS = dict(objective="regression", metric="rmse", learning_rate=0.08,
              num_leaves=96, min_data_in_leaf=50, feature_fraction=0.75,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              feature_pre_filter=False, num_threads=0, verbose=-1, seed=42)

SEED_CFG = [(42, 0.75, 0.80), (79, 0.60, 0.90), (116, 0.85, 0.70),
            (153, 0.70, 0.85), (190, 0.65, 0.75)]
SEEDS = [s for s, _, _ in SEED_CFG]

def seed_params(i):
    p = dict(PARAMS)
    p["seed"], p["feature_fraction"], p["bagging_fraction"] = SEED_CFG[i]
    return p

def rmsle(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(np.asarray(y_true, float))) ** 2)))

dtr = lgb.Dataset(df.loc[tr_m, FEATURES], df.loc[tr_m, "target"],
                  categorical_feature=CATS, free_raw_data=False)
dva = lgb.Dataset(df.loc[va_m, FEATURES], df.loc[va_m, "target"],
                  categorical_feature=CATS, reference=dtr, free_raw_data=False)

t = time.time()
m0 = lgb.train(seed_params(0), dtr, num_boost_round=2400, valid_sets=[dva],
               callbacks=[lgb.early_stopping(150, verbose=False)])
BEST_ROUNDS = m0.num_trees()
print(f"  lgbm 1/5: {BEST_ROUNDS} trees  ({time.time()-t:.0f}s)")

va_logs_lgbm = [m0.predict(df.loc[va_m, FEATURES])]
for i in range(1, len(SEEDS)):
    t2 = time.time()
    mi = lgb.train(seed_params(i), dtr, num_boost_round=BEST_ROUNDS)
    va_logs_lgbm.append(mi.predict(df.loc[va_m, FEATURES]))
    print(f"  lgbm {i+1}/5  ({time.time()-t2:.0f}s)")
    del mi; gc.collect()

y_va = np.expm1(df.loc[va_m, "target"].to_numpy())
p_va_lgbm_log = np.mean(va_logs_lgbm, axis=0)
p_va_lgbm = np.clip(np.expm1(p_va_lgbm_log), 0, None)
print(f"\nLightGBM 5-seed holdout rmsle: {rmsle(y_va, p_va_lgbm):.5f}  "
      f"(production reference: 0.39081)")

  lgbm 1/5: 1654 trees  (217s)
  lgbm 2/5  (261s)
  lgbm 3/5  (195s)
  lgbm 4/5  (204s)
  lgbm 5/5  (205s)

LightGBM 5-seed holdout rmsle: 0.39012  (production reference: 0.39081)


---
## TiDE half

**Reversible instance normalisation is the key change from v1.** Series levels here span five
orders of magnitude (`GROCERY I` versus `BOOKS`). A dense network fed raw `log1p` values across
1,782 such series burns capacity learning *scale* instead of *shape*, which is the most likely
reason v1's TiDE landed at 0.507 — barely better than assuming every day looks like the same
weekday recently (0.5206).

RIN normalises each input window and **de**normalises the output before the loss is computed,
so the network sees standardised inputs while the optimised objective remains RMSLE. An
external per-series `Scaler` was deliberately *not* used instead: dividing each series by its
own σ reweights the loss by 1/σ_series and would stop optimising the competition metric.

Direct multi-step (`output_chunk_length=16`), not recursive — 16 steps of feeding predictions
back was measured to lose decisively for the GBDT here, and there is no reason to expect
otherwise for a neural net.

In [8]:
INPUT_CHUNK = 120        # was 90; now covers the 112-day rolling window the GBDT uses
OUTPUT_CHUNK = HORIZON   # 16, direct multi-step
N_EPOCHS = 60            # was 30
PATIENCE = 10            # was 5
BATCH_SIZE = 1024

fam_codes = {f: i for i, f in enumerate(sorted(SERIES.get_level_values(1).unique()))}
city_codes = {c: i for i, c in enumerate(sorted(geo.city.unique()))}
state_codes = {s: i for i, s in enumerate(sorted(geo.state.unique()))}
type_codes = {t: i for i, t in enumerate(sorted(geo.store_type.unique()))}

# Future covariates: everything known in advance for the forecast window. `onpromotion` is
# given in test.csv, and the calendar/holiday columns are computable for any date -- the same
# known-future logic the GBDT's promo leads rely on.
cal_fut = pd.DataFrame(index=full_idx)
cal_fut["dow"] = full_idx.dayofweek
cal_fut["day"] = full_idx.day
cal_fut["month"] = full_idx.month
cal_fut["dayofyear"] = full_idx.dayofyear
cal_fut["is_weekend"] = (full_idx.dayofweek >= 5).astype(int)
cal_fut["days_to_month_end"] = full_idx.days_in_month - full_idx.day
cal_fut["payday_window"] = (full_idx.day.isin([15, 16, 17, 1, 2, 3]) |
                            (cal_fut.days_to_month_end <= 1)).astype(int)
cal_fut = cal_fut.join(cal_hol.set_index("date"))
nat_code = (nat_name.set_index("date")["nat_hol_name"].reindex(full_idx)
            .astype("category").cat.codes)
cal_fut["nat_hol_code"] = nat_code.to_numpy()
cal_fut = cal_fut.astype("float32")

series_list, fut_cov_list = [], []
t = time.time()
for store, fam in SERIES:
    ts = TimeSeries.from_series(L[(store, fam)].loc[MODEL_START:TRAIN_END], freq="D")
    row = geo.loc[store]
    ts = ts.with_static_covariates(pd.DataFrame([{
        "store_nbr": float(store), "cluster": float(row.cluster),
        "family_code": float(fam_codes[fam]), "city_code": float(city_codes[row.city]),
        "state_code": float(state_codes[row.state]),
        "type_code": float(type_codes[row.store_type]),
    }]))
    series_list.append(ts)

    fut_df = cal_fut.loc[MODEL_START:].copy()
    fut_df.insert(0, "promo", P[(store, fam)].loc[MODEL_START:].astype("float32"))
    fut_cov_list.append(TimeSeries.from_dataframe(fut_df, freq="D"))

print(f"{len(series_list)} darts TimeSeries built, "
      f"{fut_cov_list[0].width} future covariates  ({time.time()-t:.0f}s)")

1782 darts TimeSeries built, 13 future covariates  (8s)


In [9]:
# darts' `val_series` is NOT just "the values to score" -- darts builds (input_window,
# output_window) training pairs out of it, so it needs at least
# input_chunk_length + output_chunk_length days, not just the 16 holdout days. Passing only the
# holdout raises "validation time series dataset is too short for obtaining even one training
# point". Slicing from INPUT_CHUNK days before VALID_START yields exactly one validation sample
# per series: the lookback window -> the 16 held-out days. The lookback overlaps training dates,
# which is correct: what must be held out is the target window, not the inputs.
train_series = [s.drop_after(VALID_START) for s in series_list]
VAL_SLICE_START = VALID_START - pd.Timedelta(days=INPUT_CHUNK)
val_series = [s.slice(VAL_SLICE_START, TRAIN_END) for s in series_list]
print(f"train ends {train_series[0].end_time():%Y-%m-%d} | "
      f"val {val_series[0].start_time():%Y-%m-%d} -> {val_series[0].end_time():%Y-%m-%d} "
      f"({len(val_series[0])} days, need >= {INPUT_CHUNK + OUTPUT_CHUNK})")

def build_tide(n_epochs, callbacks):
    return TiDEModel(
        input_chunk_length=INPUT_CHUNK, output_chunk_length=OUTPUT_CHUNK,
        hidden_size=256, num_encoder_layers=2, num_decoder_layers=2,
        decoder_output_dim=32, temporal_decoder_hidden=64,
        use_layer_norm=True, dropout=0.1,
        use_static_covariates=True,
        use_reversible_instance_norm=True,   # the key change from v1 -- see the note above
        n_epochs=n_epochs, batch_size=BATCH_SIZE,
        optimizer_kwargs={"lr": 1e-3},
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1,
                           "callbacks": callbacks, "enable_progress_bar": False},
        random_state=42,
    )

t = time.time()
tide = build_tide(N_EPOCHS, [EarlyStopping(monitor="val_loss", patience=PATIENCE, mode="min")])
tide.fit(series=train_series, future_covariates=fut_cov_list,
         val_series=val_series, val_future_covariates=fut_cov_list, verbose=False)
TIDE_TRAIN_SECS = time.time() - t
print(f"TiDE trained  ({TIDE_TRAIN_SECS:.0f}s)")

train ends 2017-07-30 | val 2017-04-02 -> 2017-08-15 (136 days, need >= 136)


/usr/lib/python3.12/contextlib.py:137: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  return next(self.gen)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/py

TiDE trained  (645s)


In [10]:
va_preds = tide.predict(n=HORIZON, series=train_series, future_covariates=fut_cov_list)

# Flatten to df.loc[va_m]'s row order, which is date-major with series varying fastest
# (built with np.repeat(dates) / np.tile(SERIES)). Stacking each series' 16 values along
# axis=1 gives (16, n_s); ravel() then walks dates outer, series inner -- matching.
p_va_tide_log = np.stack([p.values().ravel() for p in va_preds], axis=1).ravel()
p_va_tide = np.clip(np.expm1(p_va_tide_log), 0, None)

s_tide, s_lgbm = rmsle(y_va, p_va_tide), rmsle(y_va, p_va_lgbm)
resid_corr = np.corrcoef(np.log1p(p_va_tide) - np.log1p(y_va),
                         np.log1p(p_va_lgbm) - np.log1p(y_va))[0, 1]

print(f"TiDE solo holdout    : {s_tide:.5f}   (v1 was 0.50711; naive bar 0.52063)")
print(f"LightGBM 5-seed      : {s_lgbm:.5f}   (production 0.39081)")
print(f"residual correlation : {resid_corr:.3f}   (v1 0.770; XGBoost was 0.974-0.987)")
print()
if s_tide > 0.45:
    print("!! TiDE is still far weaker than LightGBM. Per this notebook's stated stopping rule, "
          "the blend will not rescue this -- expect the sweep to pick w=1.0. Treat a marginal "
          "blend gain here as noise, not a result.")
else:
    print("TiDE is in a competitive range -- the blend sweep below is worth reading.")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


TiDE solo holdout    : 0.49912   (v1 was 0.50711; naive bar 0.52063)
LightGBM 5-seed      : 0.39012   (production 0.39081)
residual correlation : 0.790   (v1 0.770; XGBoost was 0.974-0.987)

!! TiDE is still far weaker than LightGBM. Per this notebook's stated stopping rule, the blend will not rescue this -- expect the sweep to pick w=1.0. Treat a marginal blend gain here as noise, not a result.


---
## Blend weight sweep

Log-space average, matching this project's seed-ensembling convention. `w` is LightGBM's share. Reference points: the seed-noise floor on this holdout is **0.0039**, and v1's best blend gain was **+0.0002** — anything of that order is unmeasured, not small.

In [11]:
rows = []
for w in (1.0, 0.95, 0.9, 0.85, 0.8, 0.7, 0.6, 0.5, 0.3, 0.0):
    s = rmsle(y_va, np.clip(np.expm1(w * p_va_lgbm_log + (1 - w) * p_va_tide_log), 0, None))
    rows.append((w, s))

sweep = pd.DataFrame(rows, columns=["w_lgbm", "rmsle"])
sweep["vs_lgbm_alone"] = sweep.rmsle - s_lgbm
print(sweep.to_string(index=False, float_format=lambda v: f"{v: .5f}"))

BLEND_W = float(sweep.loc[sweep.rmsle.idxmin(), "w_lgbm"])
gain = s_lgbm - sweep.rmsle.min()
print(f"\nbest w(lgbm) = {BLEND_W}   gain vs LightGBM alone = {gain:+.5f}")
if BLEND_W == 1.0:
    print("=> TiDE adds nothing. Submitting this is submitting the LightGBM model.")
elif gain < 0.0039:
    print(f"=> gain {gain:.5f} is below the 0.0039 seed-noise floor: UNMEASURED, not small. "
          "Do not treat this as a win without a paired repeat.")
else:
    print("=> gain exceeds the noise floor -- a genuine ensemble result, worth submitting.")

  w_lgbm    rmsle  vs_lgbm_alone
 1.00000  0.39012        0.00000
 0.95000  0.39062        0.00050
 0.90000  0.39171        0.00159
 0.85000  0.39340        0.00328
 0.80000  0.39567        0.00555
 0.70000  0.40194        0.01182
 0.60000  0.41041        0.02029
 0.50000  0.42094        0.03082
 0.30000  0.44760        0.05748
 0.00000  0.49912        0.10900

best w(lgbm) = 1.0   gain vs LightGBM alone = +0.00000
=> TiDE adds nothing. Submitting this is submitting the LightGBM model.


---
## Refit on full history and write the submission

In [12]:
full_tr = (df.date <= TRAIN_END) & df.target.notna() & ~eq_row
dfull = lgb.Dataset(df.loc[full_tr, FEATURES], df.loc[full_tr, "target"],
                    categorical_feature=CATS, free_raw_data=False)
te_logs = []
for i in range(len(SEEDS)):
    t = time.time()
    mi = lgb.train(seed_params(i), dfull, num_boost_round=BEST_ROUNDS)
    te_logs.append(mi.predict(df.loc[te_m, FEATURES]))
    print(f"  lgbm refit {i+1}/5  ({time.time()-t:.0f}s)")
    del mi; gc.collect()
te_log_lgbm = np.mean(te_logs, axis=0)

  lgbm refit 1/5  (207s)
  lgbm refit 2/5  (266s)
  lgbm refit 3/5  (200s)
  lgbm refit 4/5  (208s)
  lgbm refit 5/5  (203s)


In [13]:
# v1 BUG, fixed here: the refit was configured `n_epochs=tide.epochs_trained`, which returned
# 0 -> "Trainer.fit stopped: max_epochs=0", the refit finished in 1 second, and an UNTRAINED
# network produced the submitted predictions. Never read an epoch count back off a fitted model
# to configure a refit; fall back to the configured budget.
REFIT_EPOCHS = max(1, int(getattr(tide, "epochs_trained", 0) or 0) or N_EPOCHS)
print(f"refitting TiDE for {REFIT_EPOCHS} epochs on full history")

t = time.time()
tide_full = build_tide(REFIT_EPOCHS, [])
tide_full.fit(series=series_list, future_covariates=fut_cov_list, verbose=False)
refit_secs = time.time() - t
te_preds = tide_full.predict(n=HORIZON, series=series_list, future_covariates=fut_cov_list)
te_log_tide = np.stack([p.values().ravel() for p in te_preds], axis=1).ravel()
print(f"TiDE refit + predict  ({refit_secs:.0f}s)")

# Guard 1: an untrained net "trains" in seconds. If the refit was far faster than the original
# fit, it did not train.
assert refit_secs > 0.2 * TIDE_TRAIN_SECS, (
    f"refit took {refit_secs:.0f}s vs {TIDE_TRAIN_SECS:.0f}s for the original fit -- it almost "
    "certainly did not train (check for 'max_epochs=0'). This is exactly the v1 failure.")

# Guard 2: an untrained net predicts ~0 in log space, far off the LightGBM scale.
print(f"mean log prediction -- lgbm {te_log_lgbm.mean():.3f} | tide {te_log_tide.mean():.3f}")
assert abs(te_log_tide.mean() - te_log_lgbm.mean()) < 1.0, (
    "TiDE's predictions are wildly off the LightGBM scale -- the refit did not train properly. "
    "Blending this would corrupt the submission.")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


refitting TiDE for 60 epochs on full history


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=60` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


TiDE refit + predict  (6764s)
mean log prediction -- lgbm 3.638 | tide 3.628


In [14]:
pred = np.clip(np.expm1(BLEND_W * te_log_lgbm + (1 - BLEND_W) * te_log_tide), 0, None)

out = df.loc[te_m, ["date", "store_nbr", "family"]].copy()
out["sales"] = pred
out["family"] = out.family.astype(str)

key = test.assign(family=test.family.astype(str))[["id", "date", "store_nbr", "family"]]
submission = key.merge(out, on=["date", "store_nbr", "family"], how="left")

assert len(submission) == len(test)
assert submission.sales.notna().all()
assert (submission.sales >= 0).all()
assert submission.id.equals(test.id)

# Guard 3 -- the one that would have caught v1 on its own, in seconds, without needing to
# understand the bug at all. v1 shipped 6,382,939 units against a validated 12,932,032 and
# nobody read the number before uploading.
volume = submission.sales.sum()
ratio = volume / VALIDATED_VOLUME
print(f"total predicted units : {volume:,.0f}")
print(f"validated reference   : {VALIDATED_VOLUME:,.0f}   (ratio {ratio:.3f})")
assert 0.85 < ratio < 1.15, (
    f"predicted volume is {ratio:.2f}x the validated model's. Something is wrong -- v1's "
    "untrained-refit bug produced exactly this signature (ratio 0.49). Do not submit.")
print("volume check passed")

submission[["id", "sales"]].to_csv(OUT / "submission.csv", index=False)
print(f"\nwritten -> {(OUT / 'submission.csv').resolve()}")
print(f"blend weight used: w(lgbm)={BLEND_W}")

total predicted units : 12,921,062
validated reference   : 12,932,032   (ratio 0.999)
volume check passed

written -> /kaggle/working/submission.csv
blend weight used: w(lgbm)=1.0


---
## Reading this result

Compare against the two reference points before deciding anything:

| Reference | Value |
|---|---|
| Production LightGBM, this holdout | **0.39081** |
| Production LightGBM, real leaderboard | **0.42074** |
| Seed-noise floor on this holdout | **0.0039** |
| v1's blend gain (with a correctly trained TiDE) | **+0.0002** |

**Submit only if the blend gain exceeds the noise floor.** A gain of 0.001–0.002 is not a small
win here; it is an unmeasured one, and this project has three separate near-adoptions on
margins that size which later proved to be noise. If `w(lgbm)` came back as 1.0, the sweep is
telling you the blend is the LightGBM model with extra steps — and the correct action is to
record the null result rather than upload it.

If TiDE's solo score did land in a competitive range and the blend gain is real, the next step
before trusting it is a **paired repeat across the 2015/2016 position-matched backtests**, which
is where this project's local↔leaderboard agreement has been most reliable.